# 04 · Evasion cost

**Weeks 3–5. The contribution.**

Two models with statistically indistinguishable clean accuracy can demand
wildly different attacker effort. Adding a manifest permission is free and
breaks nothing; restructuring API call sequences costs real engineering.

For the linear SVM this is near closed-form — greedy flip order is optimal
because contributions are additive with no interaction term. For LightGBM,
greedy re-scoring gives an upper bound on true minimum cost, which is the safe
direction: it can only understate evadability.

Long loop → fully resumable. Safe to lose the session.


In [ ]:
# Bootstrap -- see environment/colab_bootstrap.md for the full version
import os
os.environ.setdefault('AFS_DATA_ROOT', '/content/drive/MyDrive/afs-data')
os.environ.setdefault('AFS_SCRATCH', '/content/afs-scratch')
from afs.paths import Paths
from afs.config import resolve_experiment
from afs.pipeline import run_experiment
paths = Paths.create()
print('data root:', paths.data_root)


In [ ]:
res = run_experiment(resolve_experiment('04_evasion_cost'), paths)
import pandas as pd
rows = [{'arm': a, 'roc_auc': v['roc_auc'], 'tpr@1e-3': v['tpr@fpr=0.001'],
         **{k: v.get('evasion', {}).get(k) for k in
            ['evasion_rate','median_cost','median_flips','p10_cost']}}
        for a, v in res['arms'].items()]
display(pd.DataFrame(rows).round(4))

### The money plot

Clean performance on one axis, attacker cost on the other. Arms clustering vertically — same accuracy, different cost — *is* the paper.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4.5))
for r in rows:
    if r['median_cost'] is None: continue
    ax.scatter(r['roc_auc'], r['median_cost'], s=90)
    ax.annotate(r['arm'], (r['roc_auc'], r['median_cost']),
                textcoords='offset points', xytext=(6,4), fontsize=9)
ax.set_xlabel('ROC AUC (clean)'); ax.set_ylabel('Median evasion cost')
ax.set_title('Equal accuracy, unequal security')
ax.grid(alpha=.3); fig.tight_layout()
fig.savefig('paper/figures/accuracy_vs_evasion_cost.pdf')

### Sensitivity

A reviewer will ask whether the conclusion is an artifact of the cost numbers in `docs/decisions/0005`. Answer it before they ask.


In [ ]:
# Sweep remove_penalty and the P:S cost ratio; confirm the ordering is stable.
# TODO: implement once real data is in place.